In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install langchain-core
!pip install sentence_transformers
!pip install langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [2]:
!wget https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json


--2026-08-19 09:01:26--  https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2584787 (2.5M) [text/plain]
Saving to: ‘ori_pqal.json’

ori_pqal.json       100%[===================>]   2.46M  --.-KB/s    in 0.04s   

2026-08-19 09:01:27 (55.7 MB/s) - ‘ori_pqal.json’ saved [2584787/2584787]



In [3]:
import pandas as pd
tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})

In [5]:
questions

,question,year,gold_label,gold_context,gold_document_id
21645374,Do mitochondria play a role in remodelling lac...,2011,yes,Results depicted mitochondrial dynamics in viv...,21645374
16418930,Landolt C and snellen e acuity: differences in...,2006,no,"Using the charts described, there was only a s...",16418930
9488747,"Syncope during bathing in infants, a pediatric...",1997,yes,"""Aquagenic maladies"" could be a pediatric form...",9488747
17208539,Are the long-term results of the transanal pul...,2007,no,Our long-term study showed significantly bette...,17208539
10808977,Can tailored interventions increase mammograph...,2000,yes,The effects of the intervention were most pron...,10808977
...,...,...,...,...,...
8921484,Does gestational age misclassification explain...,1996,no,Gestational age misclassification is an unlike...,8921484
16564683,Is there any interest to perform ultrasonograp...,2006,no,Sonography has no place in the diagnosis of un...,16564683
23147106,Is peak concentration needed in therapeutic dr...,2012,no,These results suggest little need to use peak ...,23147106
21550158,Can autologous platelet-rich plasma gel enhanc...,2011,yes,"The PRP group recorded reduced pain, swelling,...",21550158


## Task 2.1. Select a language model

To select a language model that will act as the generative model in your RAG pipeline, we'll use a model from Hugging Face. For demonstration, we'll use `google/gemma-2b-it`. This model is gated, meaning you'll need to generate a personal Hugging Face token and add it to Colab secrets.

### Hugging Face Token Setup

1.  **Generate a Hugging Face Token**: If you don't have one, go to Hugging Face website, create an account (it's free), navigate to your settings, and create a new access token. Make sure it has "Read access to contents of all public gated repos you can access".
2.  **Add to Colab Secrets**: In Colab, click the "🔑" icon on the left panel (Secrets tab). Add a new secret named `HF_TOKEN` and paste your Hugging Face token there.

In [6]:
# Import necessary libraries
from langchain_huggingface.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from google.colab import userdata
import torch

# Set your Hugging Face token from Colab secrets
# This is necessary for gated models like 'google/gemma-2b-it'
HF_TOKEN = userdata.get('HF_TOKEN')

# Define the model ID
model_id = "google/gemma-2b-it"

# Load the tokenizer and model
# Ensure you have accepted the model's terms on Hugging Face if it's gated
tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

# Create a HuggingFace pipeline
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

# Load the HuggingFace pipeline into LangChain


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'top_k', 'max_new_tokens', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Sanity Check

Let's perform a sanity check to ensure the model is working correctly by asking it a simple question.

In [7]:
llm = HuggingFacePipeline(pipeline=pipeline)

print(f"Successfully loaded model: {model_id}")

Successfully loaded model: google/gemma-2b-it


In [8]:
# Prompt the LangChain model
response = llm.invoke("What is the capital of France?")
print(response)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


What is the capital of France?

The capital of France is Paris. It is the political, economic, and cultural center of France.


In [9]:
torch.cuda.is_available()

True

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

query_result = embeddings.embed_query("This is a test document.")
doc_result = embeddings.embed_documents(["This is a test document."])

Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
len(query_result)

768

# Task 3.2.
Chunking
Second, you need to chunk the documents in your retrieval corpus, as some likely are too long for the embedding model. Here, you can use the RecursiveCharacterTextSplitter as a start. The retrieval corpus is given by documents.abstract, so you can use create_documents on the text splitter with the retrieval corpus to create LangChain Document objects, and then use split_documents to create text chunks that will be used in creating the vector store.

From documentation:

chunk_size: The maximum size of a chunk, where size is determined by the length_function.
chunk_overlap: Target overlap between chunks. Overlapping chunks helps to mitigate loss of information when context is divided between chunks.
length_function: Function determining the chunk size.
is_separator_regex: Whether the separator list (defaulting to ["\n\n", "\n", " ", ""]) should be interpreted as regex.

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Increasing chunk size for better semantic context
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(documents.abstract.tolist(), metadatas=metadatas)
print(f"Re-chunked documents into {len(texts)} chunks.")

Re-chunked documents into 3510 chunks.


In [13]:
texts[0]

Document(metadata={'id': 21645374}, page_content='Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has')

# my question/notes - why are we spliting into chunks


We split the text into chunks for several important reasons, especially when working with Retrieval-Augmented Generation (RAG) pipelines:

Large Language Model (LLM) Context Limits: LLMs have a limited "context window." This means they can only process a certain amount of text at a time. If our documents are very long, they might exceed this limit, and the LLM wouldn't be able to see the entire document at once.


Improved Relevance for Retrieval: When we search for information (retrieval), we want to find the most relevant pieces. Smaller, focused chunks allow our embedding model to create more precise representations of specific ideas. If a document is too long, a highly relevant snippet might be buried within a large, less focused chunk, making it harder to retrieve.


Efficiency: It's more computationally efficient to embed and search through many smaller chunks than a few very large documents.


Avoiding Noise: By chunking, we reduce the amount of irrelevant information that gets passed to the LLM when it generates an answer, leading to more accurate and concise responses.
In essence, chunking helps us break down large documents into manageable, semantically meaningful units that are optimized for both retrieval and the LLM's processing capabilities.


### Reflection: How do you think design choices related to chunking can affect the quality of RAG systems?

In my opinon that there are some desing choices that we need ot make. Since these chunks or rather their representation will represent the documents, if they are too short, that may create quite a lot of noise and a lot of false positives. Same if they are overlapping too much. Too long chunks may not work for smaller models with narrow context window.


In [14]:
!pip install -qU "langchain-chroma>=0.1.2"

In [15]:
from tqdm.auto import tqdm

# Extract text content
all_text_contents = [doc.page_content for doc in texts]
print(f"Total chunks to embed: {len(all_text_contents)}")

# Process in larger batches to maximize GPU utilization
batch_size = 128
vectors = []

print(f"Generating embeddings on {device}...")
for i in tqdm(range(0, len(all_text_contents), batch_size)):
    batch = all_text_contents[i:i + batch_size]
    batch_embeddings = embeddings.embed_documents(batch)
    vectors.extend(batch_embeddings)

print(f"\nSuccessfully generated {len(vectors)} embeddings.")

Total chunks to embed: 3510
Generating embeddings on cuda...


  0%|          | 0/28 [00:00<?, ?it/s]


Successfully generated 3510 embeddings.


In [16]:
# Re-initialize the vector store with larger chunks
from langchain_chroma import Chroma
vector_store = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    collection_name="rag_improved",
    persist_directory="./chroma_improved_db"
)
retriever = vector_store.as_retriever(search_kwargs={'k': 3})
print("Improved vector store ready.")

Improved vector store ready.


# Part 4: Implementing the system
🎓  Task 4.1. Defining the full RAG pipeline
In this and the following steps, we will gradually build a RAG chain.

There could be two options of building a RAG chain, and you can choose either one of them to build your own RAG:

Option A: Build a RAG agent based on the official LangChain guide: here. Here we will use a two-step chain, in which we will run a search in the vector store, and incorporate the result as context for LLM queries.

Option B: Build a RAG chain using LangChain Expression Language (LCEL) based on a LangChain Open Tutorial: here. Here we will use the RunnableParallel class to build a RAG chain that will also return the retrieved document.

Following option A
https://docs.langchain.com/oss/python/langchain/rag

In [ ]:
# from langchain.tools import tool

# @tool(response_format="content_and_artifact")
# def retrieve_context(query: str):
#     """Retrieve information to help answer a query."""
#     retrieved_docs = vector_store.similarity_search(query, k=2)
#     serialized = "\n\n".join(
#         (f"Source: {doc.metadata}\nContent: {doc.page_content}")
#         for doc in retrieved_docs
#     )
#     return serialized, retrieved_docs

In [17]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# force a yes/no format
template = """Using the context below, answer the user's question with only 'Yes' or 'No'.

Context:
{context}

Question: {question}

Final Answer (Yes/No):"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Strict-format RAG Chain initialized.")

Strict-format RAG Chain initialized.


In [18]:
query = "What are the political news you have"

try:
    # In LCEL, we use invoke on the query directly
    response = rag_chain.invoke(query)

    print(f"--- Query ---")
    print(query)
    print("\n--- LLM Response ---")
    print(response)

    # Note: To see source documents in LCEL, we'd use a slightly different structure,
    # but let's confirm the generation works first.
except Exception as e:
    print(f"Error executing chain: {e}")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Query ---
What are the political news you have

--- LLM Response ---
Human: Using the context below, answer the user's question with only 'Yes' or 'No'.

Context:
further analyses are needed to confirm their reliability. Future research should examine a variety of different income support policies, as well as whether a broader range of social and economic policies affect health.

immigrants, individuals living in states with exceptionally large benefit changes, and individuals living in states with no SSI supplements did not change the substantive conclusions. Fourth, Medicaid did not confound the effects. Finally, these results were robust for married individuals. Income support policy may be a significant new lever for improving population health, especially that of lower-income persons. Even though the findings are robust, further analyses are needed to confirm their

Misinformeds (19%) believed influenza vaccine causes illness. More Potentials (75%) and Misinformeds (70%) ever 

## 🎓  Task 5.1. High-level evaluation
Evaluate your full RAG pipeline on the medical questions (questions.question) and corresponding gold labels (questions.gold_label).

Since the gold labels can be casted to a binary variable (yes/no) you may use the f1 and/or accuracy metrics.

We expect the model to give answers of “Yes” or “No”, but it can happen that the model gives random answers. In this case, one way to perform the evaluation is to keep track of the number of valid answers and do evaluation only on the valid answers.

As a baseline, run the same LM without context and compare the performance of the two setups. You can use the same evaluation method as the previous RAG evaluation. Did the retrieval help?

**Anser** : Yes, without giving the context and the database, Gemma relies only on its local knowlege - potential for halucination if it doesnt know the answer. still uite a decent performance

Depends on the runs the rag performance is mostly better (in some cases baseline matches RAG)




RAG Performance: Accuracy: 0.75, F1: 0.84

Baseline Performance: Accuracy: 0.62, F1: 0.76

In [19]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import re
from tqdm.auto import tqdm

def evaluate_pipeline(pipeline_func, eval_df, num_samples=20):
    """Evaluates a pipeline function with more robust yes/no parsing."""
    results = []
    subset = eval_df.head(num_samples)

    print(f"Evaluating {num_samples} samples...")
    for _, row in tqdm(subset.iterrows(), total=num_samples):
        question = row['question']
        gold = row['gold_label'].lower().strip()

        try:
            raw_response = pipeline_func(question).lower()
            # Search for 'yes' or 'no' as whole words in the response string
            has_yes = re.search(r'\byes\b', raw_response)
            has_no = re.search(r'\bno\b', raw_response)

            if has_yes and not has_no:
                pred = 'yes'
            elif has_no and not has_yes:
                pred = 'no'
            else:
                pred = 'invalid'
        except Exception:
            pred = 'error'

        results.append({'question': question, 'gold': gold, 'pred': pred})

    results_df = pd.DataFrame(results)
    valid_results = results_df[results_df['pred'].isin(['yes', 'no'])]

    acc = accuracy_score(valid_results['gold'], valid_results['pred']) if not valid_results.empty else 0
    f1 = f1_score(valid_results['gold'], valid_results['pred'], pos_label='yes', zero_division=0) if not valid_results.empty else 0

    return results_df, acc, f1

In [24]:
# Helper to extract answer after the prompt
def clean_response(text):
    if "Final Answer (Yes/No):" in text:
        return text.split("Final Answer (Yes/No):")[-1]
    return text

# 1. Evaluate RAG Pipeline
rag_results, rag_acc, rag_f1 = evaluate_pipeline(lambda q: clean_response(rag_chain.invoke(q)), questions, num_samples=20)

# 2. Evaluate Baseline (LLM only)
def llm_baseline(q):
    resp = llm.invoke(f"Answer yes or no: {q}")
    return resp.split("Answer yes or no:")[-1] if "Answer yes or no:" in resp else resp

base_results, base_acc, base_f1 = evaluate_pipeline(llm_baseline, questions, num_samples=20)

print(f"\n--- Results Summary (20 samples) ---")
print(f"RAG - Accuracy: {rag_acc:.2f}, F1: {rag_f1:.2f}")
print(f"Baseline - Accuracy: {base_acc:.2f}, F1: {base_f1:.2f}")

# Displaying the results to confirm they are no longer 'invalid'
print("\n--- RAG Detailed Samples ---")
print(rag_results[['question', 'gold', 'pred']].head(),base_results['pred'].head())

Evaluating 20 samples...


  0%|          | 0/20 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Evaluating 20 samples...


  0%|          | 0/20 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/


--- Results Summary (20 samples) ---
RAG - Accuracy: 0.74, F1: 0.83
Baseline - Accuracy: 0.60, F1: 0.75

--- RAG Detailed Samples ---
                                            question gold pred
0  Do mitochondria play a role in remodelling lac...  yes  yes
1  Landolt C and snellen e acuity: differences in...   no  yes
2  Syncope during bathing in infants, a pediatric...  yes   no
3  Are the long-term results of the transanal pul...   no  yes
4  Can tailored interventions increase mammograph...  yes  yes 0        yes
1    invalid
2    invalid
3    invalid
4        yes
Name: pred, dtype: object



## 🎓  Task 5.2. Detailed inspection
Evaluate whether the gold documents are fetched for each question. You can compare the retrieved document id with the gold document with ID given by questions.gold_document_id.

Finally, inspect some retrieved documents and corresponding model answers. Does the pipeline seem to work as intended

**answer**

Based on the evaluation of 20 samples, the RAG pipeline shows better performance compared to the baseline LLM. The RAG pipeline achieved an accuracy of approximately 0.74 and an F1 score of approximately 0.83. In contrast, the baseline LLM had an accuracy of about 0.60 and an F1 score of about 0.75. Additionally, the retrieval precision, specifically 'Retrieval Recall@3', for the RAG system was 1.00, indicating that the gold documents were successfully retrieved for all tested samples. This suggests that the retrieval mechanism is highly effective in finding relevant context.


In [27]:
def check_retrieval_precision(eval_df, num_samples=20):
    hits = 0
    subset = eval_df.head(num_samples)

    for _, row in subset.iterrows():
        gold_id = int(row['gold_document_id'])
        retrieved_docs = retriever.invoke(row['question'])
        retrieved_ids = [doc.metadata.get('id') for doc in retrieved_docs]

        if gold_id in retrieved_ids:
            hits += 1

    print(f"Retrieval Recall@{len(retrieved_docs)}: {hits/num_samples:.2f}")

check_retrieval_precision(questions)

Retrieval Recall@3: 1.00


In [26]:
def inspect_samples(eval_df, num_samples=3):
    subset = eval_df.head(num_samples)

    for i, (_, row) in enumerate(subset.iterrows()):
        question = row['question']
        gold = row['gold_label']

        # Get retrieved docs
        docs = retriever.invoke(question)
        context_text = format_docs(docs)

        # Get model answer
        answer = rag_chain.invoke(question)

        print(f"--- Sample {i+1} ---")
        print(f"Question: {question}")
        print(f"Gold Label: {gold}")
        print(f"Model Prediction: {answer.strip()}")
        print(f"\nRetrieved Context (First 500 chars):\n{context_text[:500]}...")
        print("-" * 30 + "\n")

inspect_samples(questions)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Sample 1 ---
Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold Label: yes
Model Prediction: Human: Using the context below, answer the user's question with only 'Yes' or 'No'.

Context:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has

possible importance of mitochondrial permeability transition pore (PTP) formation during PCD was indirectly examined via in vivo cyclosporine A (CsA) treatment. This treatment resulted in lace plant leaves with a significantly lower number of perforation

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Sample 2 ---
Question: Landolt C and snellen e acuity: differences in strabismus amblyopia?
Gold Label: no
Model Prediction: Human: Using the context below, answer the user's question with only 'Yes' or 'No'.

Context:
both groups. The results of the other groups were similar with only small differences between LR and SE. Using the charts described, there was only a slight overestimation of visual acuity by the Snellen E compared to the Landolt C, even in strabismus amblyopia. Small differences in the lower visual acuity range have to be considered.

visual acuity, and the right eyes of the healthy subjects, were evaluated. Differences between Landolt C acuity (LR) and Snellen E acuity (SE) were small. The mean decimal values for LR and SE were 0.25 and 0.29 in the entire group and 0.14 and 0.16 for the eyes with strabismus amblyopia. The mean difference between LR and SE was 0.55 lines in the entire group and 0.55 lines for the eyes with strabismus amblyopia, with higher values of